In [1]:
!pip3 install icalendar pandas requests


[notice] A new release of pip available: 22.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import requests
import pandas as pd

all_holidays = []

for year in range(2019, 2027):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/NZ"
    data = requests.get(url).json()
    for h in data:
        all_holidays.append({
            "year": year,
            "date": h["date"],
            "holiday_name": h["localName"],
            "english_name": h["name"],
            "country": h["countryCode"],
            "global": h["global"]
        })

df = pd.DataFrame(all_holidays)
df.to_csv("new_zealand_holidays_2019_2026.csv", index=False)
df.to_csv("new_zealand_holidays_2019_2026.csv", index=False)

df.head()

,year,date,holiday_name,english_name,country,global
0,2019,2019-01-01,New Year's Day,New Year's Day,NZ,True
1,2019,2019-01-02,Day after New Year's Day,Day after New Year's Day,NZ,True
2,2019,2019-01-21,Wellington Anniversary Day,Wellington Anniversary Day,NZ,False
3,2019,2019-01-28,Auckland/Northland Anniversary Day,Auckland Anniversary Day,NZ,False
4,2019,2019-02-04,Nelson Anniversary Day,Nelson Anniversary Day,NZ,False


In [7]:
import requests
import pandas as pd

# -----------------------------
# PUBLIC HOLIDAYS
# -----------------------------
all_holidays = []

for year in range(2019, 2027):

    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/NZ"
    data = requests.get(url).json()

    for h in data:
        all_holidays.append({
            "date": h["date"],
            "holiday_name": h["localName"],
            "type": "Public Holiday"
        })

# -----------------------------
# WEEKENDS
# -----------------------------
dates = pd.date_range(start="2019-01-01", end="2026-12-31")

weekends = []

for d in dates:

    # Saturday = 5, Sunday = 6
    if d.weekday() in [5, 6]:

        weekends.append({
            "date": d.strftime("%Y-%m-%d"),
            "holiday_name": "Weekend",
            "type": "Weekend"
        })

# -----------------------------
# COMBINE BOTH
# -----------------------------
df_public = pd.DataFrame(all_holidays)
df_weekend = pd.DataFrame(weekends)

df = pd.concat([df_public, df_weekend], ignore_index=True)

# Remove duplicates
df = df.drop_duplicates(subset=["date"])

# Sort by date
df = df.sort_values("date")

# Add year column
df["year"] = pd.to_datetime(df["date"]).dt.year

# Save CSV
df.to_csv("nz_holidays_with_weekends_2019_2026.csv", index=False)

# -----------------------------
# COUNT TOTAL HOLIDAYS
# -----------------------------
print("Total holidays including weekends:", len(df))

print(df.head())

Total holidays including weekends: 1004
           date              holiday_name            type  year
0    2019-01-01            New Year's Day  Public Holiday  2019
1    2019-01-02  Day after New Year's Day  Public Holiday  2019
182  2019-01-05                   Weekend         Weekend  2019
183  2019-01-06                   Weekend         Weekend  2019
184  2019-01-12                   Weekend         Weekend  2019


In [8]:
# Total number of holidays
total_holidays = len(df)

print("Total Holidays:", total_holidays)

Total Holidays: 1004


In [9]:
import requests
import pandas as pd

# ---------------------------------
# GET NZ PUBLIC HOLIDAYS
# ---------------------------------
holiday_dates = []

for year in range(2019, 2027):

    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/NZ"
    data = requests.get(url).json()

    for h in data:
        holiday_dates.append(h["date"])

# Convert to set
holiday_dates = set(holiday_dates)

# ---------------------------------
# CREATE ALL DATES
# ---------------------------------
df = pd.DataFrame({
    "date": pd.date_range(
        start="2019-01-01",
        end="2026-12-31"
    )
})

# Format date
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

# ---------------------------------
# WEEKEND CHECK
# ---------------------------------
weekend_mask = pd.to_datetime(df["date"]).dt.weekday.isin([5, 6])

# ---------------------------------
# PUBLIC HOLIDAY CHECK
# ---------------------------------
public_holiday_mask = df["date"].isin(holiday_dates)

# ---------------------------------
# FINAL ONE-HOT ENCODING
# 1 = Holiday
# 0 = Not Holiday
# ---------------------------------
df["holiday"] = (
    weekend_mask | public_holiday_mask
).astype(int)

# ---------------------------------
# SAVE CSV
# ---------------------------------
df.to_csv("nz_holidays_onehot.csv", index=False)

# ---------------------------------
# CHECK OUTPUT
# ---------------------------------
print(df.head(20))

print("\nTotal Holidays:")
print(df["holiday"].sum())

          date  holiday
0   2019-01-01        1
1   2019-01-02        1
2   2019-01-03        0
3   2019-01-04        0
4   2019-01-05        1
5   2019-01-06        1
6   2019-01-07        0
7   2019-01-08        0
8   2019-01-09        0
9   2019-01-10        0
10  2019-01-11        0
11  2019-01-12        1
12  2019-01-13        1
13  2019-01-14        0
14  2019-01-15        0
15  2019-01-16        0
16  2019-01-17        0
17  2019-01-18        0
18  2019-01-19        1
19  2019-01-20        1

Total Holidays:
1004


In [11]:
!pip3 install xarray netCDF4 rasterio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 11.2 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 19.1 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.7/490.7 kB 11.0 MB/s eta 0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 19.4 MB/s eta 0:00:00

[notice] A new release of pip available: 22.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [31]:
import requests
import pandas as pd

# NASA POWER hourly API
url = "https://power.larc.nasa.gov/api/temporal/hourly/point"

# Parameters
params = {
    "parameters": "ALLSKY_SFC_SW_DWN,T2M,CLOUD_AMT",
    "community": "RE",
    "longitude": 174.7633,
    "latitude": -36.8485,
    "start": "20190101",
    "end": "20260531",
    "format": "JSON",
    "time-standard": "LST"
}

# Request data
response = requests.get(url, params=params)

# Convert to JSON
data = response.json()

# Check if API returned properly
print(data.keys())
print(data.get("messages", ""))

# Extract variables
solar = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
temp = data["properties"]["parameter"]["T2M"]
cloud = data["properties"]["parameter"]["CLOUD_AMT"]

# Create dataframe
df = pd.DataFrame({
    "time": list(solar.keys()),
    "solar_radiation_kwh_m2": list(solar.values()),
    "temperature_2m": list(temp.values()),
    "cloud_amount": list(cloud.values())
})

# Convert time format
df["time"] = pd.to_datetime(df["time"], format="%Y%m%d%H")

# Show data
print(df.head(50))

# Dataset size
print(df.shape)

# Save CSV
df.to_csv("nz_hourly_solar_nasa_power_2019_2026.csv", index=False)

print("CSV saved successfully!")

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])
[]
                  time  solar_radiation_kwh_m2  temperature_2m  cloud_amount
0  2019-01-01 00:00:00                    0.00           18.35         88.27
1  2019-01-01 01:00:00                    0.00           18.29         88.31
2  2019-01-01 02:00:00                    0.00           18.15         95.35
3  2019-01-01 03:00:00                    0.00           18.07         80.42
4  2019-01-01 04:00:00                    0.00           18.03         66.64
5  2019-01-01 05:00:00                   19.55           18.47         75.56
6  2019-01-01 06:00:00                  104.18           19.65         85.02
7  2019-01-01 07:00:00                  193.98           20.47         91.82
8  2019-01-01 08:00:00                  308.80           21.49         98.30
9  2019-01-01 09:00:00                  355.60           22.23         99.17
10 2019-01-01 10:00:00                  455.92           22

In [32]:
import requests
import pandas as pd

url = "https://data.niwa.co.nz/api/getStations"

params = {
    "productRef": "climate-station-hourly"
}

response = requests.get(url, params=params)

data = response.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['stations'])


In [33]:
print(data["stations"][0])

MANORBURN DAM


In [34]:
import json

print(json.dumps(data["stations"][0], indent=2))

"MANORBURN DAM"


In [35]:
stations_df = pd.DataFrame(data["stations"])

print(stations_df.head())

stations_df.to_csv("niwa_stations.csv", index=False)

                0
0   MANORBURN DAM
1   VANUATU,PEKOA
2  TERRA NOVA BAY
3  PENANG MILL-RA
4    DARGAVILLE 2


In [36]:
main_cities = [
    "Auckland",
    "Wellington",
    "Christchurch",
    "Hamilton",
    "Dunedin"
]

filtered = stations_df[
    stations_df["name"].str.contains(
        "|".join(main_cities),
        case=False,
        na=False
    )
]

print(filtered[["name", "agentNo"]])

KeyError: 'name'

In [37]:
print(stations_df.columns)

RangeIndex(start=0, stop=1, step=1)


In [38]:
print(stations_df.head())

                0
0   MANORBURN DAM
1   VANUATU,PEKOA
2  TERRA NOVA BAY
3  PENANG MILL-RA
4    DARGAVILLE 2


In [39]:
print(stations_df.iloc[0])

0    MANORBURN DAM
Name: 0, dtype: object


In [40]:
filtered = stations_df[
    stations_df["stationName"].str.contains(
        "|".join(main_cities),
        case=False,
        na=False
    )
]

KeyError: 'stationName'

In [41]:
import requests
import pandas as pd
import json

url = "https://data.niwa.co.nz/api/getStations"

params = {
    "productRef": "climate-station-hourly"
}

response = requests.get(url, params=params)

data = response.json()

# Print top-level keys
print(data.keys())

# Print first item fully
print(json.dumps(data, indent=2)[:3000])

dict_keys(['stations'])
{
  "stations": [
    "MANORBURN DAM",
    "VANUATU,PEKOA",
    "TERRA NOVA BAY",
    "PENANG MILL-RA",
    "DARGAVILLE 2",
    "WAIKERIA 2",
    "TIMARU AERO",
    "CHRISTCHURCH, MT PLEASANT",
    "LEVIN M.A.F.",
    "THAMES 2",
    "TE PUKE EDR",
    "HAMILTON AERO",
    "INVERCARGILL AERO 2 EWS",
    "ALBANY,NZMS",
    "HALVFARRYGGEN ER11",
    "PORANGAHAU 2",
    "CI RAROTONGA AERO",
    "WAIRAPUKAO FOREST",
    "HUKANUI",
    "MOLESWORTH",
    "LGB.10",
    "MT LYFORD",
    "MT COOK,TASMAN AERO",
    "MT SOMERS, SOMER DOWNS",
    "SOLOMON, AUKI",
    "HAAST",
    "APPLEBY",
    "LINCOLN RD, AUCKLAND REGIONAL COUNCIL",
    "LAKE TAHAROA",
    "NASEBY FOREST 1",
    "L74400",
    "BALMORAL",
    "OTAUTAU N.Z.F.S.",
    "LENINGRADSKAJA",
    "KAIKOURA WEATHER STN",
    "FRANZ JOSEF @ CRAWFORD KNOB",
    "WAIPARA NORTH BRANCH AT LANGS GULLY CWS",
    "TANGA AWS",
    "CI AITUTAKI AWS",
    "CROMWELL GORGE",
    "WESTPORT AERO",
    "WAIPARA WEST",
    "FIJI, UN

In [42]:
import pandas as pd

# Try converting entire response
stations_df = pd.json_normalize(data)

print(stations_df.columns)

Index(['stations'], dtype='object')


In [43]:
stations_df = pd.json_normalize(data["data"])

KeyError: 'data'

In [46]:
print(stations_df.columns)

Index(['stations'], dtype='object')


In [48]:
print(stations_df.head())

                                            stations
0  [MANORBURN DAM, VANUATU,PEKOA, TERRA NOVA BAY,...


In [52]:
print(stations_df.head())

                                            stations
0  [MANORBURN DAM, VANUATU,PEKOA, TERRA NOVA BAY,...


In [53]:
print(stations_df.columns.tolist())

['stations']


In [54]:
['stations']

['stations']

In [55]:
# Extract the station list
station_list = stations_df["stations"][0]

# Convert list into dataframe
stations_expanded = pd.DataFrame(station_list)

# Check columns
print(stations_expanded.columns)

# View first rows
print(stations_expanded.head())

RangeIndex(start=0, stop=1, step=1)
                0
0   MANORBURN DAM
1   VANUATU,PEKOA
2  TERRA NOVA BAY
3  PENANG MILL-RA
4    DARGAVILLE 2


In [56]:
main_cities = [
    "Auckland Aero",
    "Wellington Aero",
    "Christchurch Aero",
    "Hamilton Aero",
    "Dunedin Aero",
    "Tauranga Aero",
    "Palmerston North Aero",
    "Nelson Aero",
    "Rotorua Aero",
    "Invercargill Aero"
]

filtered = stations_expanded[
    stations_expanded["stationName"].str.contains(
        "|".join(main_cities),
        case=False,
        na=False
    )
]

print(filtered[[
    "stationName",
    "agentNo",
    "latitude",
    "longitude"
]])

KeyError: 'stationName'

In [57]:
print(stations_expanded.columns.tolist())

[0]


In [58]:
print(stations_expanded.head())

                0
0   MANORBURN DAM
1   VANUATU,PEKOA
2  TERRA NOVA BAY
3  PENANG MILL-RA
4    DARGAVILLE 2


In [59]:
stations_expanded

,0
0,MANORBURN DAM
1,"VANUATU,PEKOA"
2,TERRA NOVA BAY
3,PENANG MILL-RA
4,DARGAVILLE 2
...,...
1456,"Winchmore 2, Raine EWS"
1457,MURCHISON RAWS
1458,WAIAU SCHOOL CWS
1459,LEVIN AWS


In [60]:
print(stations_expanded.columns.tolist())

[0]


In [61]:
print(stations_expanded.head(20))

                            0
0               MANORBURN DAM
1               VANUATU,PEKOA
2              TERRA NOVA BAY
3              PENANG MILL-RA
4                DARGAVILLE 2
5                  WAIKERIA 2
6                 TIMARU AERO
7   CHRISTCHURCH, MT PLEASANT
8                LEVIN M.A.F.
9                    THAMES 2
10                TE PUKE EDR
11              HAMILTON AERO
12    INVERCARGILL AERO 2 EWS
13                ALBANY,NZMS
14         HALVFARRYGGEN ER11
15               PORANGAHAU 2
16          CI RAROTONGA AERO
17          WAIRAPUKAO FOREST
18                    HUKANUI
19                 MOLESWORTH


In [62]:
stations_expanded.columns = ["stationName"]

print(stations_expanded.head())

      stationName
0   MANORBURN DAM
1   VANUATU,PEKOA
2  TERRA NOVA BAY
3  PENANG MILL-RA
4    DARGAVILLE 2


In [63]:
main_cities = [
    "Auckland",
    "Wellington",
    "Christchurch",
    "Hamilton",
    "Dunedin",
    "Tauranga",
    "Palmerston",
    "Nelson",
    "Rotorua",
    "Invercargill"
]

filtered = stations_expanded[
    stations_expanded["stationName"].str.contains(
        "|".join(main_cities),
        case=False,
        na=False
    )
]

print(filtered)

                                stationName
7                 CHRISTCHURCH, MT PLEASANT
11                            HAMILTON AERO
12                  INVERCARGILL AERO 2 EWS
27    LINCOLN RD, AUCKLAND REGIONAL COUNCIL
48                         DUNEDIN AERO AWS
...                                     ...
1425                      INVERCARGILL AERO
1431                         AUCKLAND,OTARA
1435                    Auckland, Sky tower
1439                WELLINGTON, KELBURN AWS
1452                      AUCKLAND, ARDMORE

[98 rows x 1 columns]


In [64]:
requests.post(https://shopify.com/85932015932/account/customer/api/2025-07/graphql)

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1760866980.py, line 1)

In [66]:
import requests

url = "https://data.niwa.co.nz/api/getStations?productRef=climate-station-hourly"

response = requests.post(url)

print(response.status_code)
print(response.text)

403
Cross-site POST form submissions are forbidden
